In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# 1. 파일 읽기
# 한글 깨짐 방지를 위해 encoding='cp949'를 사용합니다.
input_file = '/content/drive/MyDrive/뉴스크롤링_project/data_2226_20260121.csv'
output_csv = '/content/drive/MyDrive/뉴스크롤링_project/종목명.csv'
output_json = '/content/drive/MyDrive/뉴스크롤링_project/종목명.json'

try:
    # 한글 인코딩(cp949)으로 파일 읽기
    df = pd.read_csv(input_file, encoding='cp949')

    # 2. 시장구분(KOSPI, KOSDAQ) 필터링
    df = df[df['시장구분'].isin(['KOSPI', 'KOSDAQ'])].copy()

    # 3. 시가총액 단위 변환(십억->억) 및 1000억 이상 필터링
    df['시가총액'] = df['시가총액'] * 10
    df = df[df['시가총액'] >= 2000]

    # 4. "스팩" 및 "우선주" 종목 제거

    # 4-1. "스팩" 포함된 종목 제거 (기존 로직)
    df = df[~df['종목명'].str.contains('스팩')]

    # 4-2. "우선주" 제거 로직 (끝자리가 우, 우B, 우(전환), 우C 인 경우)
    # 정규표현식 설명:
    #   우$       : '우'로 끝나는 것 (예: 삼성전자우)
    #   우B$      : '우B'로 끝나는 것 (예: 삼성물산우B)
    #   우\(전환\)$ : '우(전환)'으로 끝나는 것 (예: CJ4우(전환))
    #   우C$      : '우C'로 끝나는 것 (예: 아모레퍼시픽홀딩스3우C)

    preferred_pattern = r'(우|우B|우\(전환\)|우C)$'

    # 우선주 패턴에 해당하면서, 종목명이 '성우'가 아닌 것만 제외(remove)
    # 즉, (우선주 패턴 O) AND (성우 X) 인 것을 찾아서 제거
    is_preferred = df['종목명'].str.contains(preferred_pattern, regex=True)
    is_exception = df['종목명'] == '성우'  # '성우'는 코스닥 상장사(보통주)이므로 보호

    # (우선주이면서 예외가 아닌 것)을 제외(~)
    df = df[~(is_preferred & ~is_exception)]

    # 5. "종목명" 컬럼만 추출 (Series 객체로 변환)
    stock_names = df['종목명']

    # 6. 파일 저장
    # CSV: header=False로 설정하여 열 이름("종목명") 제거
    stock_names.to_csv(output_csv, index=False, header=False, encoding='utf-8-sig')

    # JSON: orient='values'로 설정하여 키 없이 값만 있는 리스트 ["A", "B", ...] 형태로 저장
    stock_names.to_json(output_json, orient='values', force_ascii=False, indent=4)

    print(f"변환 완료!")
    print(f"총 {len(stock_names)}개의 종목이 저장되었습니다.")
    print(f"생성된 파일: {output_csv} (헤더 없음), {output_json} (리스트 형태)")

except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다: {input_file}")
except Exception as e:
    print(f"오류가 발생했습니다: {e}")

/tmp/ipython-input-3418009528.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_preferred = df['종목명'].str.contains(preferred_pattern, regex=True)


변환 완료!
총 928개의 종목이 저장되었습니다.
생성된 파일: /content/drive/MyDrive/뉴스크롤링_project/종목명.csv (헤더 없음), /content/drive/MyDrive/뉴스크롤링_project/종목명.json (리스트 형태)
